## Data preprocessing


In [10]:
import json
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import random
import pickle
from utils import load_data, create_embedding_matrix, preprocess_text, to_padding, pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support


In [11]:
class MyTokenizer:
    def __init__(self):
        self.word_index = {"<PAD>": 0, "<UNK>": 1}
        self.idx_to_token = {0: "<PAD>", 1: "<UNK>"}
        
    def fit_on_texts(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word_index:
                    self.word_index[word] = len(self.word_index)
                    self.idx_to_token[self.word_index[word]] = word

    def text_to_sequences(self, text):
        return [self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()]
    
    def texts_to_sequences(self, texts):
        return [[self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()] for text in texts]


    def encode(self, text, max_length):
        tokens = self.text_to_sequences(text)
        if len(tokens) < max_length:
            tokens += [self.word_index["<PAD>"]] * (max_length - len(tokens))
        else:
            tokens = tokens[:max_length]
        return tokens
    
    def __call__(self, claims, evidences, max_length=512, return_tensors="pt"):
        self.fit_on_texts([claims, evidences])
        encoded_claims = self.encode(claims, max_length)
        encoded_evidences = self.encode(evidences, max_length)
        if return_tensors == "pt":
            return {
                "input_ids": torch.tensor([encoded_claims, encoded_evidences], dtype=torch.long)
            }
        return {"input_ids": [encoded_claims, encoded_evidences]}


In [12]:
# nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer =  SnowballStemmer('english')

train_claims_data = load_data('../data/train-claims.json')
evidence_data = load_data('../data/evidence.json')
dev_claims_data = load_data('../data/dev-claims.json')
# evidence_map = load_data('../data/curated/preprocessed_evidence_map.json')  
# evidence_map = load_data('../data/curated/mild_nostopwords_evidence.json')  
filtered_evidence_map = load_data('../data/curated/mild_nostopwords_filtered_evidence.json')

In [13]:
# # Process evidences
# evidence_map = {eid: preprocess_text(text, stemmer, stop_words) for eid, text in evidence_data.items()}
# with open("data/curated/mild_preprocesses_evidence_map.json", "w") as f:
#     json.dump(evidence_map, f)

In [14]:
# # Load filtered evidences
# filtered_evidences = load_data("data/curated/climate_dic.json")
# filtered_evidence_map = {eid: preprocess_text(text, stemmer, stop_words) for eid, text in filtered_evidences.items()}
# with open("data/curated/mild_nostopwords_filtered_evidence.json", "w") as f:
#     json.dump(filtered_evidence_map, f)

In [15]:
from sklearn.model_selection import train_test_split

claim_ids = []
for claim_id, claim_details in train_claims_data.items():
	claim_ids.append(claim_id)

# split the claims_df into training and test sets
train, test = train_test_split(claim_ids, test_size=0.2, random_state=42)
len(train)

982

In [16]:
len(train_claims_data)

1228

In [17]:
# train_data_for_dataframe = []
# test_data_for_dataframe = []
# evidence_keys = list(filtered_evidence_map.keys())  # List of all evidence IDs
# filtered_evidence_keys = list(filtered_evidence_map.keys())
# ratio = 10

# for claim_id, claim_details in train_claims_data.items():
# 	claim_text = preprocess_text(claim_details['claim_text'], stemmer, stop_words)
# 	claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

# 	# Add positive examples
# 	for eid in claim_evidences:
# 		evidence_text = filtered_evidence_map.get(eid, "NULL")  
# 		if evidence_text != "NULL":
# 			data = {
# 				'claim': claim_text,
# 				'evidence': evidence_text,
# 				'label': 1  # Label as relevant
# 			}
# 			if claim_id in train:
# 				train_data_for_dataframe.append(data)
# 			else:
# 				test_data_for_dataframe.append(data)

# 	# Add negative examples
# 	num_neg_samples = min(len(claim_evidences)*ratio, len(filtered_evidence_keys) - len(claim_evidences)*ratio)  # Limit the number of negative samples
# 	negative_samples = random.sample([k for k in filtered_evidence_keys if k not in claim_evidences], num_neg_samples)
# 	for eid in negative_samples:
# 		evidence_text = filtered_evidence_map[eid]
# 		data = {
# 			'claim': claim_text,
# 			'evidence': evidence_text,
# 			'label': 0  # Label as not relevant
# 		}
# 		if claim_id in train:
# 				train_data_for_dataframe.append(data)
# 		else:
# 			test_data_for_dataframe.append(data)

# train_df = pd.DataFrame(train_data_for_dataframe)
# test_df = pd.DataFrame(test_data_for_dataframe)

# train_df.to_csv('train_data_nostopwords_1-10.csv', index=False)
# test_df.to_csv('test_data_nostopwords_1-10.csv', index=False)

train_df = pd.read_csv("train_data_nostopwords_1-10.csv")
test_df = pd.read_csv("test_data_nostopwords_1-10.csv")

train_df = train_df.dropna()
test_df = test_df.dropna()

print(train_df.head(10))


                                               claim  \
0  not onli is there no scientif evid that co2 is...   
1  not onli is there no scientif evid that co2 is...   
2  not onli is there no scientif evid that co2 is...   
3  not onli is there no scientif evid that co2 is...   
4  not onli is there no scientif evid that co2 is...   
5  not onli is there no scientif evid that co2 is...   
6  not onli is there no scientif evid that co2 is...   
7  not onli is there no scientif evid that co2 is...   
8  not onli is there no scientif evid that co2 is...   
9  not onli is there no scientif evid that co2 is...   

                                            evidence  label  
0  plant can grow as much as 50 percent faster in...      1  
1  at veri high concentr 100 time atmospher conce...      1  
2  higher carbon dioxid concentr will favour affe...      1  
3  arcana record label a french classic record la...      0  
4  makhanya compet at the summer olymp where he w...      0  
5  a chemic

In [18]:
tokenizer = MyTokenizer()
x_claim, x_sents, x_labels, x_claims_word_index,  x_sents_word_index, y_claims_data, y_sents_data, y_labels = to_padding(train_df, test_df, tokenizer)

print ("x claim word index ", len(x_claims_word_index))
print ("x sent word index ", len(x_sents_word_index))

vocab_size_claims = len(x_claims_word_index) + 2  # +1 for padding, +1 for <UNK>
vocab_size_evidences = len(x_sents_word_index) + 2

Max length: 66
Max length: 262
x claim word index  46178
x sent word index  46178


In [19]:
class EvidenceDataset(Dataset):
    def __init__(self, claims, evidences, labels):
        self.claims = claims
        self.evidences = evidences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        claim = torch.tensor(self.claims[idx], dtype=torch.long)
        evidence = torch.tensor(self.evidences[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return {
            'claims': claim,
            'evidences': evidence,
            'labels': label
        }

# Assuming x_claim, y_claims_data, etc. are numpy arrays or lists of integers
train_dataset = EvidenceDataset(x_claim, x_sents, x_labels)
test_dataset = EvidenceDataset(y_claims_data, y_sents_data, y_labels)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)


## Creating the Embedding Matrix

In [20]:
word_vectors = KeyedVectors.load('../word2vec.wordvectors', mmap='r')

In [21]:
embedding_dim = 300  # dimension of word2vec vectors
(embed_matrix_claim, embed_dim_claim) = create_embedding_matrix(vocab_size_claims, word_vectors, x_claims_word_index, embedding_dim)
(embed_matrix_evidence, embed_dim_evidence) = create_embedding_matrix(vocab_size_evidences, word_vectors, x_sents_word_index, embedding_dim)

print ("embed_matrix_claim shape ", embed_matrix_claim.shape)
print ("embed_matrix_evidence shape ", embed_matrix_evidence.shape)

embed_matrix_claim shape  (46180, 300)
embed_matrix_evidence shape  (46180, 300)


## Building the LSTM Model

We will build a simple unidirectional LSTM model to compare claim and evidence embeddings.

In [22]:
# class Attention(nn.Module):
#     def __init__(self, feature_dim):
#         super(Attention, self).__init__()
#         self.feature_dim = feature_dim
#         self.att_weights = nn.Parameter(torch.Tensor(1, feature_dim), requires_grad=True)
#         nn.init.xavier_uniform_(self.att_weights.data)

#     def forward(self, x):
#         # Apply attention across the time dimension (dim=1) of the LSTM output
#         weights = torch.bmm(x, self.att_weights.transpose(1, 0).unsqueeze(0).repeat(x.size(0), 1, 1))
#         weights = torch.nn.functional.softmax(weights.squeeze(2), dim=1)
#         weighted_output = torch.bmm(weights.unsqueeze(1), x)
#         return weighted_output.squeeze(1)

class BahdanauAttention(nn.Module):
    def __init__(self, feature_dim):
        super(BahdanauAttention, self).__init__()
        self.feature_dim = feature_dim
        self.key_layer = nn.Linear(feature_dim, feature_dim, bias=False)
        self.query_layer = nn.Linear(feature_dim, feature_dim, bias=False)
        self.energy_layer = nn.Linear(feature_dim, 1, bias=False)
        nn.init.xavier_uniform_(self.key_layer.weight)
        nn.init.xavier_uniform_(self.query_layer.weight)
        nn.init.xavier_uniform_(self.energy_layer.weight)

    def forward(self, values):
        # Assume values is the output from an LSTM layer: [batch_size, seq_len, feature_dim]
        # Generate query and keys from the same LSTM output
        query = self.query_layer(values)  # [batch_size, seq_len, feature_dim]
        keys = self.key_layer(values)  # [batch_size, seq_len, feature_dim]

        # Compute energy scores using broadcasting
        energy = torch.tanh(query + keys)  # [batch_size, seq_len, feature_dim]
        energy = self.energy_layer(energy)  # [batch_size, seq_len, 1]
        energy = energy.squeeze(-1)  # [batch size, seq_len]

        # Softmax to obtain attention weights
        attention_weights = F.softmax(energy, dim=1)  # [batch_size, seq_len]
        attention_weights = attention_weights.unsqueeze(1)  # [batch_size, 1, seq_len]

        # Compute the context vector
        context = torch.bmm(attention_weights, values)  # [batch_size, 1, feature_dim]
        context = context.squeeze(1)  # [batch_size, feature_dim]

        return context, attention_weights

# class MultiHeadAttention(nn.Module):
#     def __init__(self, embed_size, heads):
#         super(MultiHeadAttention, self).__init__()
#         self.embed_size = embed_size
#         self.heads = heads
#         self.head_dim = embed_size // heads

#         assert self.head_dim * heads == embed_size, "Embed size needs to be divisible by heads"

#         self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
#         self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
#         self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
#         self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

#     def forward(self, values, keys, query, mask=None):
#         N = query.shape[0]
#         value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

#         # Split the embedding into self.heads different pieces
#         values = values.reshape(N, value_len, self.heads, self.head_dim)
#         keys = keys.reshape(N, key_len, self.heads, self.head_dim)
#         queries = query.reshape(N, query_len, self.heads, self.head_dim)

#         values = self.values(values)
#         keys = self.keys(keys)
#         queries = self.queries(queries)

#         # Einsum does matrix multiplication for query*keys for each training example
#         # with every other training example, don't be confused by einsum it's just a way to do batch matrix multiplication
#         energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
#         # Optional mask for attention to ignore certain positions (e.g., padding)
#         if mask is not None:
#             energy = energy.masked_fill(mask == 0, float("-1e20"))

#         attention = torch.softmax(energy / (self.embed_size ** (1 / 2)), dim=3)

#         out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
#             N, query_len, self.heads * self.head_dim
#         )

#         out = self.fc_out(out)
#         return out


class EvidenceModel(nn.Module):
    def __init__(self, vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence):
        super(EvidenceModel, self).__init__()
        self.embedding_claims = nn.Embedding(vocab_size_claims, embed_dim_claim)
        self.lstm_claims_1 = nn.LSTM(embed_dim_claim, 256, batch_first=True, bidirectional=True)
        self.lstm_claims_2 = nn.LSTM(256*2, 64, batch_first=True, bidirectional=True)
        self.attention_claims = BahdanauAttention(64*2) 
        
        self.embedding_evidences = nn.Embedding(vocab_size_evidences, embed_dim_evidence)
        self.lstm_evidences_1 = nn.LSTM(embed_dim_evidence, 256, batch_first=True, bidirectional=True)
        self.lstm_evidences_2 = nn.LSTM(256*2, 16, batch_first=True, bidirectional=True)
        self.attention_evidences = BahdanauAttention(16*2)  
        
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(80*2, 64)  
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

         # Load pre-trained embeddings
        self.embedding_claims.weight.data.copy_(torch.from_numpy(embed_matrix_claim))
        self.embedding_evidences.weight.data.copy_(torch.from_numpy(embed_matrix_evidence))
        
        # Optionally freeze the embeddings
        self.embedding_claims.weight.requires_grad = False
        self.embedding_evidences.weight.requires_grad = False
    
    def forward(self, claims_input, evidences_input):
        embedded_claims = self.embedding_claims(claims_input)
        lstm_out_claims, _ = self.lstm_claims_1(embedded_claims)
        lstm_out_claims, _ = self.lstm_claims_2(lstm_out_claims)
        # _, (h_claims, _) = self.lstm_claims_1(embedded_claims)
        # query_claims = h_claims[-1]
        # attentive_claims = self.attention_claims(lstm_out_claims)
        # attentive_claims, _ = self.attention_claims(query_claims, lstm_out_claims)
        attentive_claims, _ = self.attention_claims(lstm_out_claims)  # Self-attention on the output of the second LSTM
        # print("Claims attention output size:", attentive_claims.shape)

        embedded_evidences = self.embedding_evidences(evidences_input)
        lstm_out_evidences, _ = self.lstm_evidences_1(embedded_evidences)
        lstm_out_evidences, _ = self.lstm_evidences_2(lstm_out_evidences)  
        attentive_evidences, _ = self.attention_evidences(lstm_out_evidences)  
        # _, (h_evidences, _) = self.lstm_evidences_1(embedded_evidences)
        # query_evidences = h_evidences[-1]
        # # attentive_evidences = self.attention_evidences(lstm_out_evidences)
        # attentive_evidences, _ = self.attention_evidences(query_evidences, lstm_out_evidences)
        
        # print("Evidences attention output size:", attentive_evidences.shape)

        concatenated = torch.cat((attentive_claims, attentive_evidences), dim=1)
        # print("Concatenated size:", concatenated.shape)
        
        concatenated = self.dropout(concatenated)
        concatenated = self.fc1(concatenated)
        concatenated = self.relu(concatenated)
        output = self.fc2(concatenated)
        output = self.sigmoid(output)
        return output


def train_model(model, train_loader, val_loader, epochs, device):
    model_path = '../checkpoints/BAttn_blstm2_evidence_retrieval_1-10.pth'
    best_val_loss = float('inf')
    patience = 5
    trigger_times = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, device)
        print(f'Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

def predict(model, data_loader, device):
    model.eval() 
    predictions = []
    with torch.no_grad():  
        for batch in data_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            outputs = model(claims, evidences)
            predictions.append(outputs.cpu())  # Move predictions to CPU
            
    # Concatenate the list of tensors into a single tensor
    predictions = torch.cat(predictions, dim=0)
    return predictions


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EvidenceModel(vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence).to(device)
# model = model = ClaimEvidenceModel(vocab_size_claims+vocab_size_evidences, embedding_dim, lstm_dim=128)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
# optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.BCELoss()

In [24]:

# Train the model
epochs = 60
train_model(model, train_loader, test_loader, epochs, device)

Epoch 1, Train Loss: 0.3799, Val Loss: 0.2545


KeyboardInterrupt: 

## Test model

In [ ]:
# Load the best model
# model.load_state_dict(torch.load('lstm_evidence_retrieval.pth'))
# model.load_state_dict(torch.load('model/BAttn_lstm2_evidence_retrieval_1-10.pth'))

# Test loader setup like train_loader and val_loader
test_loss = evaluate(model, test_loader, device)
print("Test loss", test_loss)
threshold = 0.7
# Prediction and performance metrics
y_pred = []
y_true = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        claims = batch['claims'].to(device)
        evidences = batch['evidences'].to(device)
        labels = batch['labels'].cpu().numpy()
        outputs = model(claims, evidences).cpu().numpy() > threshold
        y_pred.extend(outputs)
        y_true.extend(labels)

# Calculate precision, recall, and F-score
precision, recall, fscore, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
print("Score of LSTM", {'precision': precision, 'recall': recall, 'fscore': fscore})


Test loss 0.19947805064065116
Score of LSTM {'precision': 0.7988826815642458, 'recall': 0.7096774193548387, 'fscore': 0.7516425755584758}


## Predict dev-claims

In [ ]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'], stemmer, stop_words)
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'original_claim': claim_details['claim_text'],
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

,claim_id,claim,original_claim,evidence
0,claim-752,south australia has the most expens electr in ...,[South Australia] has the most expensive elect...,"[evidence-67732, evidence-572512]"
1,claim-375,when 3 per cent of total annual global emiss o...,when 3 per cent of total annual global emissio...,"[evidence-996421, evidence-1080858, evidence-2..."
2,claim-1266,this mean that the world is now 1c warmer than...,This means that the world is now 1C warmer tha...,"[evidence-889933, evidence-694262]"
3,claim-871,as it happen zika may also be a good model of ...,"“As it happens, Zika may also be a good model ...","[evidence-422399, evidence-702226, evidence-28..."
4,claim-2164,greenland has onli lost a tini fraction of it ...,Greenland has only lost a tiny fraction of its...,"[evidence-52981, evidence-264761, evidence-947..."
...,...,...,...,...
149,claim-2400,sudden label co2 as a pollut is a disservic to...,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-409365, evidence-127519, evidence-85..."
150,claim-204,after a natur orbit driven warm atmospher carb...,"after a natural orbitally driven warming, atmo...","[evidence-368192, evidence-261690, evidence-20..."
151,claim-1426,mani of the world s coral reef are alreadi bar...,Many of the world’s coral reefs are already ba...,"[evidence-1124018, evidence-995813, evidence-1..."
152,claim-698,a recent studi led by lawrenc livermor nation ...,A recent study led by Lawrence Livermore Natio...,[evidence-660755]


In [ ]:
with open('tokenizer_claims.pickle', 'rb') as handle:
	claims_tokenizer = pickle.load(handle)

with open('tokenizer_evidence.pickle', 'rb') as handle:
	sents_tokenizer = pickle.load(handle)

max_claims_length = 70
max_sents_length = 180

In [ ]:
dev_claims_text = [preprocess_text(claim, stemmer, stop_words) for claim in dev_claims_df["claim"].tolist()]
dev_sents_text = filtered_evidence_map.values()

# Tokenize the preprocessed text
dev_claims = claims_tokenizer.texts_to_sequences(dev_claims_text)
dev_sents = sents_tokenizer.texts_to_sequences(dev_sents_text)

# Padding sequences to ensure uniform input size
dev_claims = pad_sequences(dev_claims, maxlen=max_claims_length)
dev_sents = pad_sequences(dev_sents, maxlen=max_sents_length)

# Print shapes
print("dev claims shape:", dev_claims.shape)
print("dev sents shape:", dev_sents.shape)

dev claims shape: (154, 70)
dev sents shape: (298688, 180)


In [ ]:

# model = model.to(device)  
# model.eval()  

# dev_claims = torch.tensor(dev_claims).to(device)  
# dev_sents = torch.tensor(dev_sents).to(device)
# claim_ids = dev_claims_df["claim_id"].tolist()
# evidence_ids = list(filtered_evidence_map.keys())

# batch_size = 1024  
# # threshold = 0.98
# top_k = 5
# all_predictions = []

# for i in range(dev_claims.shape[0]):
#     claim_row = dev_claims[i].unsqueeze(0)
#     claim_id = claim_ids[i]
#     # qualified_evidence_ids = []
#     #======================================================
#     top_evidence_ids = []
#     # Aggregate scores and corresponding evidence IDs across all batches
#     all_scores = []
#     all_batch_evidence_ids = []
#     #======================================================

#     # Process in batches
#     for j in range(0, dev_sents.shape[0], batch_size):
#         end = min(j + batch_size, dev_sents.shape[0])
#         batch_sents = dev_sents[j:end].to(device)
#         # replicated_claims_batch = replicated_claims[:end-j].to(device)  
#         replicated_claims_batch = claim_row.repeat(batch_sents.shape[0], 1).to(device)

#         with torch.no_grad():
#             outputs = model(replicated_claims_batch, batch_sents).squeeze(1)
#             #======================================================
#             all_scores.extend(outputs.cpu().numpy())  # Collect scores
#             all_batch_evidence_ids.extend(evidence_ids[j:end])
#             #======================================================
#             # predictions = outputs > threshold
#             # # print("Y_PREDICT: ", outputs.cpu().numpy())

#             # # Collect evidence IDs for this batch where predictions are true
#             # true_indices = predictions.cpu().nonzero(as_tuple=False).squeeze(1)
#             # batch_evidence_ids = evidence_ids[j:end]  
#             # qualified_evidence_ids.extend([batch_evidence_ids[idx] for idx in true_indices])
#             #======================================================

#     #======================================================
#     # Store results for this claim
#     if all_scores:
#         # Get indices of the top k scores
#         top_indices = sorted(range(len(all_scores)), key=lambda x: all_scores[x], reverse=True)[:top_k]
#         top_evidence_ids = [all_batch_evidence_ids[index] for index in top_indices]

#     # Store results for this claim
#     if top_evidence_ids:
#         all_predictions.append({
#             "claim_id": claim_id,
#             "evidences_id": top_evidence_ids
#         })
#     #======================================================
#     # if qualified_evidence_ids:
#     #     all_predictions.append({
#     #         "claim_id": claim_id,
#     #         "evidences_id": qualified_evidence_ids
#     #     })
#     # break

# results_df = pd.DataFrame(all_predictions)


In [ ]:
# model.load_state_dict(torch.load('../checkpoints/BAttn_lstm2_evidence_retrieval_1-10.pth'))
model = model.to(device)  
model.eval()  

dev_claims = torch.tensor(dev_claims).to(device)  
dev_sents = torch.tensor(dev_sents).to(device)
claim_ids = dev_claims_df["claim_id"].tolist()
evidence_ids = filtered_evidence_keys

batch_size = 1024  
threshold = 0.7
# top_k = 5
all_predictions = []

for i in range(dev_claims.shape[0]):
    claim_row = dev_claims[i].unsqueeze(0)
    claim_id = claim_ids[i]
    qualified_evidence_ids = []
    #======================================================
    # top_evidence_ids = []
    # # Aggregate scores and corresponding evidence IDs across all batches
    # all_scores = []
    # all_batch_evidence_ids = []
    #======================================================

    # Process in batches
    for j in range(0, dev_sents.shape[0], batch_size):
        end = min(j + batch_size, dev_sents.shape[0])
        batch_sents = dev_sents[j:end].to(device)
        # replicated_claims_batch = replicated_claims[:end-j].to(device)  
        replicated_claims_batch = claim_row.repeat(batch_sents.shape[0], 1).to(device)

        with torch.no_grad():
            outputs = model(replicated_claims_batch, batch_sents).squeeze(1)
            #======================================================
            # all_scores.extend(outputs.cpu().numpy())  # Collect scores
            # all_batch_evidence_ids.extend(evidence_ids[j:end])
            #======================================================
            predictions = outputs > threshold
            # print("Y_PREDICT: ", outputs.cpu().numpy())

            # Collect evidence IDs for this batch where predictions are true
            true_indices = predictions.cpu().nonzero(as_tuple=False).squeeze(1)
            batch_evidence_ids = evidence_ids[j:end]  
            qualified_evidence_ids.extend([batch_evidence_ids[idx] for idx in true_indices])
            #======================================================

    #======================================================
    # # Store results for this claim
    # if all_scores:
    #     # Get indices of the top k scores
    #     top_indices = sorted(range(len(all_scores)), key=lambda x: all_scores[x], reverse=True)[:top_k]
    #     top_evidence_ids = [all_batch_evidence_ids[index] for index in top_indices]

    # # Store results for this claim
    # if top_evidence_ids:
    #     all_predictions.append({
    #         "claim_id": claim_id,
    #         "evidences_id": top_evidence_ids
    #     })
    #======================================================
    if qualified_evidence_ids:
        all_predictions.append({
            "claim_id": claim_id,
            "evidences_id": qualified_evidence_ids
        })


results_df = pd.DataFrame(all_predictions)
results_df.to_csv("../data/output/BAttn_blstm_result_t70_10.csv", index=False)


In [ ]:
# model.load_state_dict(torch.load('model/BAttn_lstm2_evidence_retrieval_1-10.pth'))
model = model.to(device)  
model.eval()  

dev_claims = torch.tensor(dev_claims).to(device)  
dev_sents = torch.tensor(dev_sents).to(device)
claim_ids = dev_claims_df["claim_id"].tolist()
evidence_ids = list(filtered_evidence_map.keys())

batch_size = 1024  
threshold = 0.5
# top_k = 5
all_predictions = []

for i in range(dev_claims.shape[0]):
    claim_row = dev_claims[i].unsqueeze(0)
    claim_id = claim_ids[i]
    qualified_evidence_ids = []
    #======================================================
    # top_evidence_ids = []
    # # Aggregate scores and corresponding evidence IDs across all batches
    # all_scores = []
    # all_batch_evidence_ids = []
    #======================================================

    # Process in batches
    for j in range(0, dev_sents.shape[0], batch_size):
        end = min(j + batch_size, dev_sents.shape[0])
        batch_sents = dev_sents[j:end].to(device)
        # replicated_claims_batch = replicated_claims[:end-j].to(device)  
        replicated_claims_batch = claim_row.repeat(batch_sents.shape[0], 1).to(device)

        with torch.no_grad():
            outputs = model(replicated_claims_batch, batch_sents).squeeze(1)
            #======================================================
            # all_scores.extend(outputs.cpu().numpy())  # Collect scores
            # all_batch_evidence_ids.extend(evidence_ids[j:end])
            #======================================================
            predictions = outputs > threshold
            # print("Y_PREDICT: ", outputs.cpu().numpy())

            # Collect evidence IDs for this batch where predictions are true
            true_indices = predictions.cpu().nonzero(as_tuple=False).squeeze(1)
            batch_evidence_ids = evidence_ids[j:end]  
            qualified_evidence_ids.extend([batch_evidence_ids[idx] for idx in true_indices])
            #======================================================

    #======================================================
    # # Store results for this claim
    # if all_scores:
    #     # Get indices of the top k scores
    #     top_indices = sorted(range(len(all_scores)), key=lambda x: all_scores[x], reverse=True)[:top_k]
    #     top_evidence_ids = [all_batch_evidence_ids[index] for index in top_indices]

    # # Store results for this claim
    # if top_evidence_ids:
    #     all_predictions.append({
    #         "claim_id": claim_id,
    #         "evidences_id": top_evidence_ids
    #     })
    #======================================================
    if qualified_evidence_ids:
        all_predictions.append({
            "claim_id": claim_id,
            "evidences_id": qualified_evidence_ids
        })

results_df = pd.DataFrame(all_predictions)
results_df.to_csv("../data/output/BAttn_lstm2_result_t50_10.csv", index=False)


C:\Users\Clare\AppData\Local\Temp\ipykernel_40192\3929062036.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dev_claims = torch.tensor(dev_claims).to(device)
C:\Users\Clare\AppData\Local\Temp\ipykernel_40192\3929062036.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dev_sents = torch.tensor(dev_sents).to(device)


In [ ]:
# model.load_state_dict(torch.load('model/BAttn_lstm2_evidence_retrieval_1-10.pth'))
model = model.to(device)  
model.eval()  

dev_claims = torch.tensor(dev_claims).to(device)  
dev_sents = torch.tensor(dev_sents).to(device)
claim_ids = dev_claims_df["claim_id"].tolist()
evidence_ids = list(filtered_evidence_map.keys())

batch_size = 1024  
threshold = 0.3
# top_k = 5
all_predictions = []

for i in range(dev_claims.shape[0]):
    claim_row = dev_claims[i].unsqueeze(0)
    claim_id = claim_ids[i]
    qualified_evidence_ids = []
    #======================================================
    # top_evidence_ids = []
    # # Aggregate scores and corresponding evidence IDs across all batches
    # all_scores = []
    # all_batch_evidence_ids = []
    #======================================================

    # Process in batches
    for j in range(0, dev_sents.shape[0], batch_size):
        end = min(j + batch_size, dev_sents.shape[0])
        batch_sents = dev_sents[j:end].to(device)
        # replicated_claims_batch = replicated_claims[:end-j].to(device)  
        replicated_claims_batch = claim_row.repeat(batch_sents.shape[0], 1).to(device)

        with torch.no_grad():
            outputs = model(replicated_claims_batch, batch_sents).squeeze(1)
            #======================================================
            # all_scores.extend(outputs.cpu().numpy())  # Collect scores
            # all_batch_evidence_ids.extend(evidence_ids[j:end])
            #======================================================
            predictions = outputs > threshold
            # print("Y_PREDICT: ", outputs.cpu().numpy())

            # Collect evidence IDs for this batch where predictions are true
            true_indices = predictions.cpu().nonzero(as_tuple=False).squeeze(1)
            batch_evidence_ids = evidence_ids[j:end]  
            qualified_evidence_ids.extend([batch_evidence_ids[idx] for idx in true_indices])
            #======================================================

    #======================================================
    # # Store results for this claim
    # if all_scores:
    #     # Get indices of the top k scores
    #     top_indices = sorted(range(len(all_scores)), key=lambda x: all_scores[x], reverse=True)[:top_k]
    #     top_evidence_ids = [all_batch_evidence_ids[index] for index in top_indices]

    # # Store results for this claim
    # if top_evidence_ids:
    #     all_predictions.append({
    #         "claim_id": claim_id,
    #         "evidences_id": top_evidence_ids
    #     })
    #======================================================
    if qualified_evidence_ids:
        all_predictions.append({
            "claim_id": claim_id,
            "evidences_id": qualified_evidence_ids
        })

results_df = pd.DataFrame(all_predictions)
results_df.to_csv("../data/output/BAttn_lstm2_result_t30_10.csv", index=False)


C:\Users\Clare\AppData\Local\Temp\ipykernel_40192\1731877717.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dev_claims = torch.tensor(dev_claims).to(device)
C:\Users\Clare\AppData\Local\Temp\ipykernel_40192\1731877717.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dev_sents = torch.tensor(dev_sents).to(device)


In [ ]:
import ast
results_df = pd.read_csv("../data/output/BAttn_blstm_result_t70_10.csv")
results_df['evidences_id'] = results_df['evidences_id'].apply(ast.literal_eval)

In [ ]:
results_df

,claim_id,evidences_id
0,claim-752,"[evidence-315, evidence-337, evidence-842, evi..."
1,claim-375,"[evidence-315, evidence-337, evidence-842, evi..."
2,claim-1266,"[evidence-315, evidence-337, evidence-842, evi..."
3,claim-871,"[evidence-315, evidence-337, evidence-842, evi..."
4,claim-2164,"[evidence-315, evidence-337, evidence-842, evi..."
...,...,...
149,claim-2400,"[evidence-315, evidence-337, evidence-842, evi..."
150,claim-204,"[evidence-315, evidence-337, evidence-842, evi..."
151,claim-1426,"[evidence-315, evidence-337, evidence-842, evi..."
152,claim-698,"[evidence-315, evidence-337, evidence-842, evi..."


In [ ]:
# # Function to select 2 to 10 random evidences
# def select_random_evidences(evidence_list):
#     num_to_select = random.randint(2, min(10, len(evidence_list)))  # Select 2 to 10 or the length of the list if smaller
#     return random.sample(evidence_list, num_to_select)

# # Apply the function to each row in the DataFrame
# results_df['evidences_id'] = results_df['evidences_id'].apply(select_random_evidences)


In [ ]:
for i in results_df["evidences_id"][0]:
    print(filtered_evidence_map[i])

as el niño condit start to develop dure earli 2014 sea level in western micronesia includ in water surround the island nation of palau and guam drop by feet m
the had it warmest on record in 2012
mani comment quot one email in which phil jone said that he had use mike natur trick in a 1999 graph for the world meteorolog organ to hide the declin in proxi temperatur deriv from tree ring analys when measur temperatur were actual rise
trend in global temperatur sinc januari the begin of the satellit temperatur record measur in degre celsius per decad at as octob
most of the global warm sinc the centuri is veri like due to human activ
deglaci commenc in the northern hemispher approxim year bp and in antarctica approxim year bp which is consist with evid that this was the primari sourc for an abrupt rise in the sea level year ago
major consequ apart from region cool could also includ an increas in major flood and storm a collaps of plankton stock warm or rainfal chang in the tropic or alaska

In [ ]:
merged_results = results_df.merge(dev_claims_df, on='claim_id', how='left')

In [ ]:
merged_results.drop(columns=["claim", "evidence"], inplace=True)
merged_results.rename(columns={"original_claim": "claim_text", "evidences_id": "evidences"}, inplace=True)
merged_results['claim_label']  = "SUPPORTS"
merged_results.set_index('claim_id', inplace=True)
merged_results

,evidences,claim_text,claim_label
claim_id,,,
claim-752,"[evidence-315, evidence-337, evidence-842, evi...",[South Australia] has the most expensive elect...,SUPPORTS
claim-375,"[evidence-315, evidence-337, evidence-842, evi...",when 3 per cent of total annual global emissio...,SUPPORTS
claim-1266,"[evidence-315, evidence-337, evidence-842, evi...",This means that the world is now 1C warmer tha...,SUPPORTS
claim-871,"[evidence-315, evidence-337, evidence-842, evi...","“As it happens, Zika may also be a good model ...",SUPPORTS
claim-2164,"[evidence-315, evidence-337, evidence-842, evi...",Greenland has only lost a tiny fraction of its...,SUPPORTS
...,...,...,...
claim-2400,"[evidence-315, evidence-337, evidence-842, evi...","'To suddenly label CO2 as a ""pollutant"" is a d...",SUPPORTS
claim-204,"[evidence-315, evidence-337, evidence-842, evi...","after a natural orbitally driven warming, atmo...",SUPPORTS
claim-1426,"[evidence-315, evidence-337, evidence-842, evi...",Many of the world’s coral reefs are already ba...,SUPPORTS


In [ ]:
result = merged_results.to_json(orient="index")
with open('../data/output/BAttn-blstm-t70-output.json', 'w') as f:
    f.write(result)